# 04. Baseline: logistic regression

A simple, honest baseline: class-weighted logistic regression, a threshold chosen on validation,
one final look at test. The experiment at the end checks whether the time counter `day` helps or hurts.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd

from src.features import LOGREG_CONTINUOUS, FINAL_BINARY, CATEGORICAL
from src.split import make_masks
from src.models import make_logreg
from src.evaluation import pick_threshold, metrics_at_threshold

In [ ]:
model_df = pd.read_parquet("../data/processed/model_df.parquet")

# `day` is used only in the "with day" experiment below
model_df["day"] = ((model_df["step"] - 1) // 24 + 1).astype("int16")

train_mask, val_mask, test_mask = make_masks(model_df)

y_train = model_df.loc[train_mask, "is_fraud"]
y_val = model_df.loc[val_mask, "is_fraud"]
y_test = model_df.loc[test_mask, "is_fraud"]

## Experiment: with and without `day`

Both models see the same data and the same split and differ only in the `day` feature.
The threshold is picked again on validation for each model (best precision with recall >= 80%),
then applied once to test.

In [ ]:
experiments = {
    "with day": LOGREG_CONTINUOUS + ["day"],
    "without day": LOGREG_CONTINUOUS,
}

fitted, rows = {}, []

for name, continuous in experiments.items():
    cols = continuous + FINAL_BINARY + CATEGORICAL

    model = make_logreg(continuous, FINAL_BINARY, CATEGORICAL)
    model.fit(model_df.loc[train_mask, cols], y_train)
    fitted[name] = (model, continuous)

    p_val = model.predict_proba(model_df.loc[val_mask, cols])[:, 1]
    p_test = model.predict_proba(model_df.loc[test_mask, cols])[:, 1]
    threshold = pick_threshold(p_val, y_val)

    for part, y, p in [("validation", y_val, p_val), ("test", y_test, p_test)]:
        m = metrics_at_threshold(y, p, threshold)
        m.update(model=name, part=part, threshold=round(threshold, 3))
        rows.append(m)

results = pd.DataFrame(rows)
results[["model", "part", "threshold", "precision", "recall", "f1", "pr_auc", "TP", "FN", "FP"]].round(3)

## Coefficients

In [ ]:
for name, (model, continuous) in fitted.items():
    names = model.named_steps["preprocessor"].get_feature_names_out()
    coefs = model.named_steps["classifier"].coef_[0]
    print(f"\n{name}")
    for n, c in sorted(zip(names, coefs), key=lambda x: -abs(x[1])):
        print(f"  {n:45s} {c:+.3f}")

## Why `day` hurts: a look at the scaled values

The model learns a coefficient for `day` on the training days only and extends it linearly.
Test contains days the model has never seen, so the scaled values go far beyond the training range.

In [ ]:
model, continuous = fitted["with day"]
scaler = model.named_steps["preprocessor"].named_transformers_["continuous"]
i = continuous.index("day")
coef_day = model.named_steps["classifier"].coef_[0][i]

for name, mask in [("train", train_mask), ("validation", val_mask), ("test", test_mask)]:
    scaled = (model_df.loc[mask, "day"] - scaler.mean_[i]) / scaler.scale_[i]
    print(
        f"{name:11s} scaled day: min={scaled.min():+.2f} mean={scaled.mean():+.2f} "
        f"max={scaled.max():+.2f} -> mean contribution to the logit: {coef_day * scaled.mean():+.2f}"
    )

## Conclusion

Recall dropped from about 80% on validation to 74% on test, which looked like drift. `day` is only a time counter:
in train it takes values 1-14, in test up to 31, and a linear model extrapolates it. Without `day` the validation and
test recall are close (80.5% and 83.5%) and test PR-AUC rises from 0.562 to 0.627. The mechanism is an explanation,
not something proven here; what is proven is that the model is better without the feature, so `day` is removed.

Precision should not be compared between validation and test directly: the fraud share is very different between
the two sets, and so are the thresholds of the two models.